<a href="https://colab.research.google.com/github/jrochanav/CapstoneProject/blob/main/W7_RAG_Augmented_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
# RAG (Retrieval-Augmented Generation) for AI Applications

## Install Dependencies
!pip install faiss-cpu langchain openai chromadb python-allrecipes==0.3.1 beautifulsoup4 >= 4.6
!pip install -U langchain-community

ERROR: Could not find a version that satisfies the requirement 4.6 (from versions: none)
ERROR: No matching distribution found for 4.6


In [26]:
%%writefile app.py
import streamlit as st
import requests
import json
import re
from langchain.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Setup Hugging Face LLM with increased max length for longer outputs
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_length=1024)  # Increased max length
llm = HuggingFacePipeline(pipeline=pipe)

# Spoonacular API Key
API_KEY = "3d0c595d085844038377acbd8bd27a3c"

def search_recipes(query):
    """Search for recipes using the Spoonacular API."""
    url = "https://api.spoonacular.com/recipes/complexSearch"
    params = {
        "query": query,
        "apiKey": API_KEY,
        "number": 5,
        "instructionsRequired": True,
        "fillIngredients": True,
        "addRecipeInformation": True,
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        if 'results' in data:
            st.session_state['result_count'] = len(data['results'])
        else:
            st.session_state['result_count'] = 0

        if 'results' in data and data['results']:
            for recipe in data['results']:
                if recipe.get('analyzedInstructions') and 'extendedIngredients' in recipe:
                    return recipe
            return data['results'][0]
        else:
            return None
    except Exception as e:
        st.session_state['api_error'] = str(e)
        return None

def get_recipe_details(recipe_id):
    """Get detailed information about a recipe by ID."""
    url = f"https://api.spoonacular.com/recipes/{recipe_id}/information"
    params = {
        "apiKey": API_KEY,
        "includeNutrition": False,
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        return data
    except Exception as e:
        st.session_state['recipe_details_error'] = str(e)
        return None

def extract_steps_from_instructions(instructions):
    """Extract steps from analyzedInstructions format."""
    steps = []
    for instruction in instructions:
        for step in instruction.get('steps', []):
            steps.append(step.get('step', ''))
    return steps

def format_recipe_markdown(recipe_name, ingredients, instructions, summary=None, source=None):
    """Format recipe as markdown for direct display."""
    markdown = f"# {recipe_name}\n\n"

    if summary:
        markdown += f"{summary}\n\n"

    markdown += "## Ingredients\n\n"
    for ing in ingredients:
        markdown += f"* {ing}\n"

    markdown += "\n## Instructions\n\n"
    for i, step in enumerate(instructions, 1):
        markdown += f"{i}. {step}\n"

    if source:
        markdown += f"\n*Recipe source: {source}*"

    return markdown

def generate_recipe(query):
    """Generates a recipe based on the query, using Spoonacular API."""
    # Search for recipes based on the user's query
    recipe_search = search_recipes(query)

    if not recipe_search:
        return f"Sorry, no recipes found for '{query}'. Please try another ingredient or dish."

    # Get recipe ID
    recipe_id = recipe_search.get('id')

    # Get more detailed recipe information
    recipe_details = get_recipe_details(recipe_id) if recipe_id else None

    # Use the most detailed information available
    recipe = recipe_details if recipe_details else recipe_search

    # Extract recipe details
    recipe_name = recipe.get('title', 'Unknown Recipe')
    recipe_url = recipe.get('sourceUrl', f"https://spoonacular.com/recipes/{recipe_name.replace(' ', '-')}-{recipe_id}")

    # Get ingredients
    ingredients_list = []
    if 'extendedIngredients' in recipe:
        ingredients_list = [ingredient.get('original', '') for ingredient in recipe['extendedIngredients']]

    # Get instructions
    instructions = []
    if 'analyzedInstructions' in recipe and recipe['analyzedInstructions']:
        instructions = extract_steps_from_instructions(recipe['analyzedInstructions'])
    elif 'instructions' in recipe and recipe['instructions']:
        clean_instructions = re.sub('<.*?>', '', recipe['instructions'])
        instructions = [s.strip() for s in re.split(r'\.|\n', clean_instructions) if s.strip()]

    # Get summary if available
    summary = ""
    if 'summary' in recipe and recipe['summary']:
        summary = re.sub('<.*?>', '', recipe['summary'])

    # Debug information
    st.session_state['debug_info'] = {
        'recipe_name': recipe_name,
        'ingredients_count': len(ingredients_list),
        'instruction_count': len(instructions),
        'has_summary': bool(summary),
        'recipe_url': recipe_url
    }

    # Store raw recipe data
    st.session_state['recipe_data'] = {
        'name': recipe_name,
        'ingredients': ingredients_list,
        'instructions': instructions,
        'summary': summary,
        'url': recipe_url
    }

    # If we have good data, format it directly
    if ingredients_list and instructions:
        st.session_state['fallback_used'] = False
        formatted_recipe = format_recipe_markdown(
            recipe_name,
            ingredients_list,
            instructions,
            summary=summary[:200] + "..." if len(summary) > 200 else summary,
            source=recipe_url
        )
        return formatted_recipe

    # Otherwise use LLM as fallback
    st.session_state['fallback_used'] = True

    # Create an effective prompt for the LLM
    prompt = f"""
    Create a complete recipe for '{recipe_name}'.

    Include:
    1. Recipe title
    2. A short description
    3. List of ingredients with measurements
    4. Step-by-step cooking instructions

    Known information about this recipe:
    {summary if summary else ""}

    Format your response as a complete recipe.
    """

    st.session_state['prompt_used'] = prompt
    generated_recipe = llm(prompt)

    return generated_recipe

# Streamlit App Interface
st.title("RAG Recipe Generator")

# Input field for recipe query
query = st.text_input("Enter your recipe request (ingredient or dish name):", key="query_input")

if st.button("Generate Recipe"):
    if query:
        with st.spinner("Searching for and generating recipe..."):
            # Clear previous session state
            for key in ['api_error', 'recipe_details_error', 'debug_info', 'fallback_used', 'recipe_data']:
                if key in st.session_state:
                    del st.session_state[key]

            # Generate the recipe
            recipe = generate_recipe(query)

        st.markdown(recipe)

        # Add debug information in an expandable section
        with st.expander("Debug Information"):
            if 'debug_info' in st.session_state:
                st.write("### Recipe Data:")
                st.json(st.session_state['debug_info'])

            if 'fallback_used' in st.session_state:
                st.write(f"### Fallback generation used: {st.session_state['fallback_used']}")

            if 'api_error' in st.session_state:
                st.error(f"API Error: {st.session_state['api_error']}")

            if 'recipe_details_error' in st.session_state:
                st.error(f"Recipe Details Error: {st.session_state['recipe_details_error']}")

            if 'result_count' in st.session_state:
                st.write(f"### Search results found: {st.session_state['result_count']}")

            if 'prompt_used' in st.session_state and st.session_state.get('fallback_used', False):
                st.write("### Prompt Used:")
                st.text(st.session_state['prompt_used'])

            # Show raw recipe data if available
            if 'recipe_data' in st.session_state:
                st.write("### Raw Recipe Data:")
                recipe_data = st.session_state['recipe_data']

                st.write(f"**Name:** {recipe_data['name']}")

                st.write("**Ingredients:**")
                for ing in recipe_data.get('ingredients', []):
                    st.write(f"- {ing}")

                st.write("**Instructions:**")
                for i, step in enumerate(recipe_data.get('instructions', []), 1):
                    st.write(f"{i}. {step}")
    else:
        st.warning("Please enter a recipe request.")

# Add a note about the app
st.markdown("---")
st.markdown("""
### About this App
This app demonstrates Retrieval Augmented Generation (RAG) by combining recipe data from the Spoonacular API with a language model to generate complete recipes.
""")

Overwriting app.py


In [27]:
!pip install pyngrok streamlit

In [31]:
from pyngrok import ngrok
from google.colab import userdata

ngrok_token = userdata.get('ngrok_authtoken')
# Set up ngrok
!ngrok authtoken {ngrok_token}

public_url = ngrok.connect(8501)
print(f"Public URL: {public_url}")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Public URL: NgrokTunnel: "https://0287-35-199-14-58.ngrok-free.app" -> "http://localhost:8501"


In [32]:
!streamlit run app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.199.14.58:8501

/usr/local/lib/python3.11/dist-packages/langchain/llms/__init__.py:549: LangChainDeprecationWarning: Importing LLMs from langchain is deprecated. Importing from langchain will no longer be supported as of langchain==0.2.0. Please import from langchain-community instead:

`from langchain_community.llms import HuggingFacePipeline`.

To install langchain-community run `pip install -U langchain-community`.
  warnings.warn(
2025-03-08 04:04:48.685429: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1741406688.735903   23421 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:17414